# Laboratorio 1 — Ejercicio 1

Hallar todas las raíces no negativas de

$$f(x)=\sin(2x)-\frac{x^3}{10}+\frac{x}{2}=0$$

con una precisión de **8 cifras decimales**.

## Resumen del ejercicio:

El problema pide identificar todas las soluciones de la ecuación en el dominio $x\geq0$, no solamente producir una aproximación aislada. Por ello, la solución combina dos tareas distintas: primero se estudia el signo de la función para determinar dónde pueden existir raíces; después se aplica un algoritmo numérico con una cota de error controlada.

El análisis demuestra que existen exactamente dos raíces no negativas. La primera es la raíz exacta $x=0$. La segunda se encuentra en $[\pi/2,\sqrt{5}]$ y se aproxima por bisección como $x=1.74488744$. El método necesitó 27 iteraciones y terminó con una cota de error de $4.957\times10^{-9}$, menor que la tolerancia requerida para reportar ocho decimales.

## Objetivo y referencias

Se utilizará el método de bisección, estudiado en clase, porque garantiza convergencia cuando la función es continua y existe un cambio de signo, también los criterios de paro y las salvaguardas. También se consultó la comparación entre métodos cerrados y abiertos que se vió en clase.

El trabajo se divide en tres partes: localizar todas las raíces, aproximar la raíz positiva y comprobar que no existen otras raíces no negativas.

## 1. Definición de la función

In [1]:
import math

def f(x):
    return math.sin(2*x) - x**3/10 + x/2

### Explicación de esta parte:

La función `f(x)` traduce directamente la expresión matemática al lenguaje Python. Se importa únicamente el módulo estándar `math` para evaluar el seno y las constantes necesarias, por tanto, el notebook no depende de paquetes externos. Esta definición se reutiliza en todas las evaluaciones y en cada iteración, evitando repetir la ecuación y reduciendo el riesgo de inconsistencias.

El objetivo de esta sección todavía no es calcular una raíz, sino construir una representación computacional confiable de la ecuación que se desea resolver.

## 2. Aislamiento de las raíces

Primero, $f(0)=0$, por lo que **$x=0$ es una raíz exacta**.

Para aislar una raíz positiva se consideran los extremos $a=\pi/2$ y $b=\sqrt{5}$. En ese intervalo la función es continua y cambia de signo.

In [2]:
print(f"f(0)       = {f(0.0): .12f}")
print(f"f(pi/2)    = {f(math.pi/2): .12f}")
print(f"f(sqrt(5)) = {f(math.sqrt(5)): .12f}")
print(f"f(3)       = {f(3.0): .12f}")

f(0)       =  0.000000000000
f(pi/2)    =  0.397819704894
f(sqrt(5)) = -0.971277798961
f(3)       = -1.479415498199


### Interpretación de las evaluaciones:

La salida confirma primero que $f(0)=0$, así que esa solución no requiere aproximación numérica. Luego se observa que $f(\pi/2)=0.397819704894>0$ y $f(\sqrt{5})=-0.971277798961<0$. Como $f$ es continua, el Teorema del Valor Intermedio garantiza al menos una raíz entre esos dos puntos.

La evaluación en $x=3$ no se usa como extremo del método, se incluye como evidencia adicional de que la función continúa siendo negativa después del intervalo seleccionado. La demostración posterior completa este argumento para todo $x\geq0$.

## 3. Método de bisección

En cada iteración se calcula $c=(a+b)/2$ y se conserva el subintervalo que mantiene el cambio de signo. Se usa como cota del error la mitad del ancho del intervalo.

Para asegurar el redondeo a ocho decimales se fija $\varepsilon=0.5\times10^{-8}$. Además, se incluye un máximo de 100 iteraciones como salvaguarda.

In [3]:
def biseccion(funcion, a, b, tolerancia=5e-9, max_iteraciones=100):
    fa = funcion(a)
    fb = funcion(b)

    if fa == 0:
        return a, []
    if fb == 0:
        return b, []
    if fa * fb > 0:
        raise ValueError("El intervalo no presenta un cambio de signo.")

    historial = []

    for iteracion in range(1, max_iteraciones + 1):
        c = (a + b) / 2
        fc = funcion(c)
        cota_error = (b - a) / 2
        historial.append((iteracion, a, b, c, fc, cota_error))

        if fc == 0 or cota_error <= tolerancia:
            return c, historial

        if fa * fc < 0:
            b = c
        else:
            a = c
            fa = fc

    raise RuntimeError("No se alcanzó la convergencia.")

### Explicación del algoritmo implementado:

La función `biseccion` recibe la función matemática, los extremos $a$ y $b$, la tolerancia y un límite de iteraciones. Antes de iterar comprueba si algún extremo ya es una raíz y verifica que $f(a)f(b)<0$, sin este cambio de signo no se podría aplicar la garantía de Bolzano.

Dentro del ciclo se calcula el punto medio $c$. Si $f(a)$ y $f(c)$ tienen signos opuestos, la raíz permanece en $[a,c]$, luego en caso contrario permanece en $[c,b]$. De esta manera, el ancho del intervalo se divide entre dos en cada paso.

El historial conserva $(k,a,b,c,f(c),\text{error})$ para documentar el proceso. La columna `error` es $(b-a)/2$, una cota matemática para la distancia entre el punto medio y la raíz. El máximo de 100 iteraciones evita un ciclo indefinido si las condiciones se modificaran o el método se utilizara incorrectamente.

## 4. Cálculo de las raíces

In [4]:
tolerancia = 0.5e-8
raiz_positiva, historial = biseccion(
    f, math.pi/2, math.sqrt(5), tolerancia
)
raices = [0.0, raiz_positiva]

for indice, raiz in enumerate(raices, start=1):
    print(f"Raíz {indice}: x = {raiz:.8f}    |f(x)| = {abs(f(raiz)):.3e}")

print(f"Iteraciones de bisección para la raíz positiva: {len(historial)}")
print(f"Cota final del error: {historial[-1][5]:.3e}")

Raíz 1: x = 0.00000000    |f(x)| = 0.000e+00
Raíz 2: x = 1.74488744    |f(x)| = 6.414e-09
Iteraciones de bisección para la raíz positiva: 27
Cota final del error: 4.957e-09


### Interpretación del resultado numérico:

La lista final contiene dos valores. `0.00000000` es una raíz exacta, mientras que `1.74488744` es la aproximación producida por bisección. El residuo $|f(x)|=6.414\times10^{-9}$ indica que, al sustituir la aproximación en la ecuación, el resultado queda extremadamente cerca de cero.

La cota final $4.957\times10^{-9}$ es menor que $0.5\times10^{-8}$. Esto justifica el redondeo de la raíz positiva a ocho posiciones decimales. Las 27 iteraciones son coherentes con la convergencia lineal de bisección ya que el intervalo inicial se reduce a la mitad en cada paso hasta alcanzar la escala solicitada.

### Primeras y últimas iteraciones

Se muestran únicamente diez filas para mantener la salida legible.

In [5]:
print(f"{'k':>3} {'a':>13} {'b':>13} {'c':>13} {'f(c)':>13} {'error':>13}")

for fila in historial[:5] + historial[-5:]:
    k, a, b, c, fc, error = fila
    print(f"{k:3d} {a:13.9f} {b:13.9f} {c:13.9f} {fc:13.3e} {error:13.3e}")

  k             a             b             c          f(c)         error
  1   1.570796327   2.236067977   1.903432152    -3.552e-01     3.326e-01
  2   1.570796327   1.903432152   1.737114239     1.784e-02     1.663e-01
  3   1.737114239   1.903432152   1.820273196    -1.715e-01     8.316e-02
  4   1.737114239   1.820273196   1.778693718    -7.730e-02     4.158e-02
  5   1.737114239   1.778693718   1.757903979    -2.982e-02     2.079e-02
 23   1.744887393   1.744887551   1.744887472    -7.316e-08     7.931e-08
 24   1.744887393   1.744887472   1.744887432     1.778e-08     3.965e-08
 25   1.744887432   1.744887472   1.744887452    -2.769e-08     1.983e-08
 26   1.744887432   1.744887452   1.744887442    -4.953e-09     9.913e-09
 27   1.744887432   1.744887442   1.744887437     6.414e-09     4.957e-09


### Cómo leer la tabla:

Cada fila representa un intervalo que todavía contiene la raíz. Las columnas $a$ y $b$ son sus extremos, $c$ es el punto medio evaluado y $f(c)$ determina cuál mitad se conserva. Un valor positivo de $f(c)$ y uno negativo no representan fallos: ya que ambos son necesarios para mantener la raíz encerrada entre puntos de signo contrario.

En la primera fila la cota del error es aproximadamente $3.326\times10^{-1}$. En la última es $4.957\times10^{-9}$. La disminución sistemática muestra la propiedad central de bisección porque el error máximo se reduce aproximadamente por un factor de dos en cada iteración, aunque la aproximación pueda alternar entre ambos lados de la raíz.

## 5. ¿Por qué estas son todas las raíces no negativas?

1. **En $x=0$:** $f(0)=0$.
2. **Si $0<x\leq\pi/2$:** $\sin(2x)\geq0$ y $x(1/2-x^2/10)>0$, por tanto, $f(x)>0$ y no hay otra raíz.
3. **Si $\pi/2<x<\sqrt{5}$:** hay un cambio de signo. Además, $f'(x)=2\cos(2x)-3x^2/10+1/2<0$ en todo el intervalo, así que $f$ es estrictamente decreciente y la raíz es única.
4. **Si $\sqrt{5}\leq x\leq3$:** tanto $\sin(2x)$ como $x(1/2-x^2/10)$ son no positivos, y al menos uno es negativo, entonces $f(x)<0$.
5. **Si $x\geq3$:** $f(x)\leq1+x/2-x^3/10$. Esta cota vale $-0.2$ en $x=3$ y continúa decreciendo, por lo que tampoco puede haber raíces.

Así se cubre todo el dominio $x\geq0$ y se demuestra que las dos raíces encontradas son exhaustivas.

## Resultado:

Las raíces no negativas, expresadas con ocho cifras decimales, son

$$\boxed{x_1=0.00000000},\qquad \boxed{x_2=1.74488744}. $$

La raíz positiva se obtuvo mediante bisección en 27 iteraciones, partiendo del intervalo $[\pi/2,\sqrt{5}]$, con una cota final del error de $4.957\times10^{-9}$.

### Conclusión:

El procedimiento no se limita a encontrar un valor que haga pequeña a la función. El análisis de signos demuestra que no existen otras raíces en el dominio solicitado, mientras que la cota propia de bisección certifica la precisión de la raíz aproximada. Por ello, el resultado satisface tanto la condición de exhaustividad como la de exactitud decimal del enunciado.